In [45]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
DATASET_PATH = "dataset"

In [46]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="rgb"
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="rgb"
)

Found 4344 files belonging to 13 classes.
Using 3476 files for training.
Found 4344 files belonging to 13 classes.
Using 868 files for validation.


In [47]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [48]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [49]:
class_names = sorted([d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))])
num_classes = len(class_names)

base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))

x = layers.Rescaling(1./255)(inputs)
x = data_augmentation(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

In [50]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [51]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 118s 929ms/step - accuracy: 0.2549 - loss: 2.8280 - val_accuracy: 0.7327 - val_loss: 1.1936
Epoch 2/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 88s 805ms/step - accuracy: 0.5929 - loss: 1.3124 - val_accuracy: 0.8641 - val_loss: 0.5420
Epoch 3/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 86s 790ms/step - accuracy: 0.7195 - loss: 0.9109 - val_accuracy: 0.8940 - val_loss: 0.3619
Epoch 4/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 86s 793ms/step - accuracy: 0.7699 - loss: 0.7345 - val_accuracy: 0.9090 - val_loss: 0.2863
Epoch 5/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 83s 761ms/step - accuracy: 0.8113 - loss: 0.6146 - val_accuracy: 0.9205 - val_loss: 0.2511
Epoch 6/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 75s 693ms/step - accuracy: 0.8283 - loss: 0.5521 - val_accuracy: 0.9194 - val_loss: 0.2298
Epoch 7/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 73s 671ms/step - accuracy: 0.8452 - loss: 0.4890 - val_accuracy: 0.9309 - val_loss: 0.2164
Epoch 8/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 75s 687ms/step - accuracy: 0.8507 - loss: 

In [52]:
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

plt.figure()
plt.plot(acc, label='train')
plt.plot(val_acc, label='val')
plt.legend()

plt.savefig("plot.png")
print("Saved plot.png")

Saved plot.png


In [53]:
model.save("animal_model.h5")

In [54]:
from tensorflow.keras.preprocessing import image

def predict(img_path):
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)[0]

    for i, name in enumerate(class_names):
        print(f"{name}: {predictions[i]:.2f}")

    print(" Final:", class_names[np.argmax(predictions)])


predict("dataset/zebra/002.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
brown_bear: 0.00
buffalo: 0.00
dolphin: 0.00
elephant: 0.00
giraffe: 0.00
horse: 0.00
kangaroo: 0.00
koala: 0.00
rhino: 0.00
sea_lion: 0.00
seal: 0.00
squirrel: 0.00
zebra: 1.00
 Final: zebra


In [59]:
from tensorflow.keras.preprocessing import image

def predict(img_path):
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)[0]

    for i, name in enumerate(class_names):
        print(f"{name}: {predictions[i]:.2f}")

    print(" Final:", class_names[np.argmax(predictions)])

# test
predict("dataset/giraffe/giraffe-0123.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
brown_bear: 0.00
buffalo: 0.00
dolphin: 0.00
elephant: 0.00
giraffe: 1.00
horse: 0.00
kangaroo: 0.00
koala: 0.00
rhino: 0.00
sea_lion: 0.00
seal: 0.00
squirrel: 0.00
zebra: 0.00
 Final: giraffe
